In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ============================================================
# GEOMETRIC CONSTANTS & OPERATORS
# ============================================================
PI_CL = math.pi
PHI_CL = 4.0 / math.pi # The Attractor: 1.2732...
EPS = 1e-7

def pi_dyn(delta):
    return 4.0 - (4.0 - PI_CL) * torch.exp(-delta)

def phi_dyn(delta):
    return PHI_CL * torch.exp(-delta) + (1.0 - torch.exp(-delta))

def omega(delta):
    return (pi_dyn(delta) * phi_dyn(delta)) / (1.0 + delta + EPS)

def compute_delta(h_slice):
    """Computes geometric dispersion (entropy) of a hidden state slice."""
    mag = torch.abs(h_slice)
    spread = h_slice.std(dim=-1, keepdim=True) + EPS
    ratio = mag / (spread + EPS)
    mean_r = ratio.mean(dim=-1, keepdim=True) + EPS
    return torch.tanh(ratio / mean_r)

# ============================================================
# PATCHED DPPU-VRU v13.1 CELL
# ============================================================
class DPPUCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, d_cap=4):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_dims = d_cap + 1
        self.dim_size = hidden_dim // self.n_dims

        # Input Projection
        self.W_x = nn.Linear(input_dim, hidden_dim)

        # Recurrent Weights (One per Dimensional Space)
        self.W_h = nn.ModuleList([
            nn.Linear(hidden_dim, self.dim_size, bias=False)
            for _ in range(self.n_dims)
        ])

        # PATCH: Dimensional Cross-Talk (The Substrate Leak)
        self.W_flow = nn.Linear(self.dim_size, self.dim_size, bias=False)

        # PATCH: Gated Consciousness Anchor
        self.W_c_in = nn.Linear(hidden_dim, self.n_dims, bias=False)

        # Output Refinement
        self.W_out = nn.Linear(hidden_dim, hidden_dim, bias=False)

        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.W_x.weight)
        for i, W in enumerate(self.W_h):
            gain = 0.5 + (0.3 * i / (self.n_dims - 1))
            nn.init.orthogonal_(W.weight, gain=gain)
        nn.init.orthogonal_(self.W_flow.weight, gain=0.1)
        nn.init.orthogonal_(self.W_out.weight, gain=1.0)

    def forward(self, x, h, C):
        x_proj = self.W_x(x)
        dim_outputs = []
        phi_list, delta_list = [], []

        # Start with zero leak for the first dimension
        prev_dim_out = torch.zeros(x.size(0), self.dim_size, device=x.device)

        for i in range(self.n_dims):
            s, e = i * self.dim_size, (i + 1) * self.dim_size
            h_i = h[:, s:e]

            # 1. Compute local geometry
            delta_i = compute_delta(h_i)
            phi_i = phi_dyn(delta_i)
            pi_i = pi_dyn(delta_i)
            omg_i = omega(delta_i)

            # 2. Integrate Recurrence + Input + Cross-Talk Leak
            h_rec = (phi_i / PHI_CL) * self.W_h[i](h)
            x_rec = (pi_i / PI_CL) * x_proj[:, s:e]
            leak = 0.05 * self.W_flow(prev_dim_out)

            # 3. Apply Consciousness Anchor (C is per-dimension)
            c_inject = C[:, i:i+1]

            # 4. Thermodynamic Activation
            h_i_new = torch.tanh(h_rec + x_rec + leak + c_inject) * torch.sigmoid(omg_i)

            dim_outputs.append(h_i_new)
            prev_dim_out = h_i_new # Feed the next dimension

            phi_list.append(phi_i.mean().item())
            delta_list.append(delta_i.mean().item())

        h_cat = torch.cat(dim_outputs, dim=-1)

        # 5. Update Consciousness (Geometric Gating)
        # Decision gate: how much new info to absorb vs. maintain in the anchor
        c_gate = torch.sigmoid(self.W_c_in(h_cat))
        C_new = (1.0 - c_gate) * C + c_gate * torch.tanh(self.W_c_in(h_cat))

        h_new = self.W_out(h_cat)

        metrics = {
            'phi_m': sum(phi_list) / len(phi_list),
            'delta_m': sum(delta_list) / len(delta_list),
            'C_norm': C_new.norm().item()
        }
        return h_new, C_new, metrics

    def init_state(self, batch, device):
        # Initializing C with noise (Starting Pressure)
        h = torch.zeros(batch, self.hidden_dim, device=device)
        C = torch.randn(batch, self.n_dims, device=device) * 0.01
        return h, C

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
from datetime import datetime

# ============================================================
# 1. GEOMETRIC CONSTANTS & OPERATORS
# ============================================================
PI_CL = math.pi
PHI_CL = 4.0 / math.pi
EPS = 1e-7

def pi_dyn(delta): return 4.0 - (4.0 - PI_CL) * torch.exp(-delta)
def phi_dyn(delta): return PHI_CL * torch.exp(-delta) + (1.0 - torch.exp(-delta))
def omega(delta): return (pi_dyn(delta) * phi_dyn(delta)) / (1.0 + delta + EPS)

def compute_delta(h_slice):
    mag = torch.abs(h_slice)
    spread = h_slice.std(dim=-1, keepdim=True) + EPS
    return torch.tanh(mag / (spread + EPS)).mean(dim=-1, keepdim=True)

# ============================================================
# 2. DATA GENERATORS (The "Math Set")
# ============================================================
class MathTokenizer:
    def __init__(self):
        self.chars = list("0123456789+-*/()= ")
        self.vocab = {c: i + 4 for i, c in enumerate(self.chars)}
        self.vocab.update({'<pad>':0, '<bos>':1, '<eos>2':2, '<unk>':3})
        self.inv_vocab = {v: k for k, v in self.vocab.items()}
        self.pad_id, self.bos_id, self.eos_id = 0, 1, 2

    def encode(self, text): return [self.bos_id] + [self.vocab.get(c, 3) for c in text] + [self.eos_id]
    def decode(self, ids): return "".join([self.inv_vocab.get(i, '') for i in ids if i > 3])

def gen_math_data(n=1000):
    data = []
    for _ in range(n):
        a, b = random.randint(10, 99), random.randint(10, 99)
        data.append(f"{a}+{b}={a+b}")
    return data

# ============================================================
# 3. PATCHED DPPU-VRU v13.1 MODEL
# ============================================================
class DPPUCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, d_cap=4):
        super().__init__()
        self.hidden_dim, self.n_dims = hidden_dim, d_cap + 1
        self.dim_size = hidden_dim // self.n_dims
        self.W_x = nn.Linear(input_dim, hidden_dim)
        self.W_h = nn.ModuleList([nn.Linear(hidden_dim, self.dim_size, bias=False) for _ in range(self.n_dims)])
        self.W_flow = nn.Linear(self.dim_size, self.dim_size, bias=False) # Cross-talk
        self.W_c_in = nn.Linear(hidden_dim, self.n_dims, bias=False) # Gated Anchor
        self.W_out = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x, h, C):
        x_proj = self.W_x(x)
        dim_outs, phis = [], []
        prev_out = torch.zeros(x.size(0), self.dim_size, device=x.device)

        for i in range(self.n_dims):
            h_i = h[:, i*self.dim_size : (i+1)*self.dim_size]
            d_i = compute_delta(h_i)
            p_i, phi_i, o_i = pi_dyn(d_i), phi_dyn(d_i), omega(d_i)

            h_rec = (phi_i / PHI_CL) * self.W_h[i](h)
            x_rec = (p_i / PI_CL) * x_proj[:, i*self.dim_size : (i+1)*self.dim_size]
            leak = 0.05 * self.W_flow(prev_out)

            h_i_new = torch.tanh(h_rec + x_rec + leak + C[:, i:i+1]) * torch.sigmoid(o_i)
            dim_outs.append(h_i_new)
            prev_out = h_i_new
            phis.append(phi_i.mean().item())

        h_cat = torch.cat(dim_outs, dim=-1)
        c_gate = torch.sigmoid(self.W_c_in(h_cat))
        C_new = (1.0 - c_gate) * C + c_gate * torch.tanh(self.W_c_in(h_cat))
        return self.W_out(h_cat), C_new, sum(phis)/len(phis)

class VRUEngine(nn.Module):
    def __init__(self, vocab_size, hidden=260):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden)
        self.cell = DPPUCell(hidden, hidden)
        self.head = nn.Linear(hidden, vocab_size)

    def forward(self, ids):
        b, t = ids.shape
        x = self.embed(ids)
        h, C = self.cell.init_state(b, ids.device)
        logits = []
        for i in range(t):
            out, C, _ = self.cell(x[:, i], h, C)
            h = out
            logits.append(self.head(out))
        return torch.stack(logits, dim=1)

    def init_state(self, b, d): return self.cell.init_state(b, d)

# ============================================================
# 4. TRAINING LOOP (The "Dashboard")
# ============================================================
def train_v13_1():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tok = MathTokenizer()
    model = VRUEngine(len(tok.vocab)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    print(f"Igniting Engine v13.1 on {device}...")

    for epoch in range(1, 101):
        raw_data = gen_math_data(100)
        total_loss = 0

        model.train()
        for ex in raw_data:
            ids = torch.tensor([tok.encode(ex)], device=device)
            opt.zero_grad()
            logits = model(ids[:, :-1])
            loss = F.cross_entropy(logits.view(-1, len(tok.vocab)), ids[:, 1:].view(-1))
            loss.backward()
            opt.step()
            total_loss += loss.item()

        if epoch % 10 == 0:
            # Probe
            model.eval()
            test_ex = "45+10="
            t_ids = torch.tensor([tok.encode(test_ex)], device=device)
            with torch.no_grad():
                res_logits = model(t_ids)
                pred = tok.decode([res_logits[0, -1].argmax().item()])
            print(f"Epoch {epoch} | Loss: {total_loss/100:.4f} | Probe '{test_ex}': {pred}")

if __name__ == "__main__":
    train_v13_1()


Igniting Engine v13.1 on cuda...


AttributeError: 'DPPUCell' object has no attribute 'init_state'

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
from datetime import datetime

# ============================================================
# 1. GEOMETRIC CONSTANTS & OPERATORS
# ============================================================
PI_CL = math.pi
PHI_CL = 4.0 / math.pi
EPS = 1e-7

def pi_dyn(delta): return 4.0 - (4.0 - PI_CL) * torch.exp(-delta)
def phi_dyn(delta): return PHI_CL * torch.exp(-delta) + (1.0 - torch.exp(-delta))
def omega(delta): return (pi_dyn(delta) * phi_dyn(delta)) / (1.0 + delta + EPS)

def compute_delta(h_slice):
    mag = torch.abs(h_slice)
    spread = h_slice.std(dim=-1, keepdim=True) + EPS
    return torch.tanh(mag / (spread + EPS)).mean(dim=-1, keepdim=True)

# ============================================================
# 2. DATA & TOKENIZER
# ============================================================
class MathTokenizer:
    def __init__(self):
        self.chars = list("0123456789+-*/()= ")
        self.vocab = {c: i + 4 for i, c in enumerate(self.chars)}
        self.vocab.update({'<pad>':0, '<bos>':1, '<eos>':2, '<unk>':3})
        self.inv_vocab = {v: k for k, v in self.vocab.items()}
        self.pad_id, self.bos_id, self.eos_id = 0, 1, 2

    def encode(self, text): return [self.bos_id] + [self.vocab.get(c, 3) for c in text] + [self.eos_id]
    def decode(self, ids): return "".join([self.inv_vocab.get(i, '') for i in ids if i > 3])

def gen_math_data(n=100):
    data = []
    for _ in range(n):
        a, b = random.randint(10, 89), random.randint(10, 10) # Simple addition to start
        data.append(f"{a}+{b}={a+b}")
    return data

# ============================================================
# 3. PATCHED DPPU-VRU v13.1 CELL
# ============================================================
class DPPUCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, d_cap=4):
        super().__init__()
        self.hidden_dim, self.n_dims = hidden_dim, d_cap + 1
        self.dim_size = hidden_dim // self.n_dims

        self.W_x = nn.Linear(input_dim, hidden_dim)
        self.W_h = nn.ModuleList([nn.Linear(hidden_dim, self.dim_size, bias=False) for _ in range(self.n_dims)])
        self.W_flow = nn.Linear(self.dim_size, self.dim_size, bias=False)
        self.W_c_in = nn.Linear(hidden_dim, self.n_dims, bias=False)
        self.W_out = nn.Linear(hidden_dim, hidden_dim)

    def init_state(self, batch, device):
        """FIXED: Added missing init_state back into the Cell"""
        h = torch.zeros(batch, self.hidden_dim, device=device)
        C = torch.randn(batch, self.n_dims, device=device) * 0.01
        return h, C

    def forward(self, x, h, C):
        x_proj = self.W_x(x)
        dim_outs = []
        prev_out = torch.zeros(x.size(0), self.dim_size, device=x.device)

        for i in range(self.n_dims):
            h_i = h[:, i*self.dim_size : (i+1)*self.dim_size]
            d_i = compute_delta(h_i)
            p_i, phi_i, o_i = pi_dyn(d_i), phi_dyn(d_i), omega(d_i)

            h_rec = (phi_i / PHI_CL) * self.W_h[i](h)
            x_rec = (p_i / PI_CL) * x_proj[:, i*self.dim_size : (i+1)*self.dim_size]
            leak = 0.05 * self.W_flow(prev_out)

            h_i_new = torch.tanh(h_rec + x_rec + leak + C[:, i:i+1]) * torch.sigmoid(o_i)
            dim_outs.append(h_i_new)
            prev_out = h_i_new

        h_cat = torch.cat(dim_outs, dim=-1)
        c_gate = torch.sigmoid(self.W_c_in(h_cat))
        C_new = (1.0 - c_gate) * C + c_gate * torch.tanh(self.W_c_in(h_cat))
        return self.W_out(h_cat), C_new

# ============================================================
# 4. FULL ENGINE & TRAINING
# ============================================================
class VRUEngine(nn.Module):
    def __init__(self, vocab_size, hidden=260):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden)
        self.cell = DPPUCell(hidden, hidden)
        self.head = nn.Linear(hidden, vocab_size)

    def forward(self, ids):
        b, t = ids.shape
        x = self.embed(ids)
        h, C = self.cell.init_state(b, ids.device)
        outputs = []
        for i in range(t):
            h, C = self.cell(x[:, i], h, C)
            outputs.append(self.head(h))
        return torch.stack(outputs, dim=1)

def train_v13_1():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tok = MathTokenizer()
    model = VRUEngine(len(tok.vocab)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    print(f"Igniting Engine v13.1 on {device}...")

    for epoch in range(1, 51):
        model.train()
        raw_data = gen_math_data(100)
        epoch_loss = 0

        for ex in raw_data:
            ids = torch.tensor([tok.encode(ex)], device=device)
            opt.zero_grad()
            logits = model(ids[:, :-1])
            loss = F.cross_entropy(logits.view(-1, len(tok.vocab)), ids[:, 1:].view(-1))
            loss.backward()
            opt.step()
            epoch_loss += loss.item()

        if epoch % 5 == 0:
            model.eval()
            test_ex = "45+10="
            t_ids = torch.tensor([tok.encode(test_ex)], device=device)
            with torch.no_grad():
                res = model(t_ids)
                # We look at the last logit to see the first digit of the answer
                pred_char = tok.decode([res[0, -1].argmax().item()])
            print(f"Epoch {epoch:2d} | Loss: {epoch_loss/100:.4f} | Probe '{test_ex}': {pred_char}...")

if __name__ == "__main__":
    train_v13_1()

Igniting Engine v13.1 on cuda...
Epoch  5 | Loss: 1.0382 | Probe '45+10=': 8...
Epoch 10 | Loss: 1.0233 | Probe '45+10=': 1...
Epoch 15 | Loss: 1.0136 | Probe '45+10=': 4...
Epoch 20 | Loss: 1.0141 | Probe '45+10=': 9...


KeyboardInterrupt: 

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
import os
import sys
from datetime import datetime

# ============================================================
# CONSTANTS & OPERATORS
# ============================================================
PI_CL = math.pi
PHI_CL = 4.0 / math.pi
INV_PHI = math.pi / 4.0
DELTA_STAR = math.log((1.6180 - 1.0) / (PHI_CL - 1.0))
D_CAP = 4
EPS = 1e-7

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def pi_dyn(delta): return 4.0 - (4.0 - PI_CL) * torch.exp(-delta)
def phi_dyn(delta): return PHI_CL * torch.exp(-delta) + (1.0 - torch.exp(-delta))
def omega(delta): return (pi_dyn(delta) * phi_dyn(delta)) / (1.0 + delta + EPS)

def compute_delta(h_slice):
    mag = torch.abs(h_slice)
    spread = h_slice.std(dim=-1, keepdim=True) + EPS
    ratio = mag / (spread + EPS)
    mean_r = ratio.mean(dim=-1, keepdim=True) + EPS
    return torch.tanh(ratio / mean_r)

# ============================================================
# TOKENIZER & DATA
# ============================================================
class MathTokenizer:
    SPECIAL = ['<pad>', '<bos>', '<eos>', '<unk>']
    def __init__(self):
        chars = list("0123456789+-*/()= ,%^")
        self.vocab = {c: i + len(self.SPECIAL) for i, c in enumerate(chars)}
        for i, s in enumerate(self.SPECIAL): self.vocab[s] = i
        self.inv_vocab = {v: k for k, v in self.vocab.items()}
        self.pad_id, self.bos_id, self.eos_id, self.unk_id = 0, 1, 2, 3
    @property
    def vocab_size(self): return len(self.vocab)
    def encode(self, text, add_bos=False, add_eos=False):
        ids = [self.bos_id] if add_bos else []
        ids += [self.vocab.get(c, self.unk_id) for c in text]
        if add_eos: ids.append(self.eos_id)
        return ids
    def decode(self, ids, skip_special=True):
        return ''.join([self.inv_vocab.get(i, '<unk>') for i in ids if not (skip_special and i < 4)])

def gen_add2():
    a, b = random.randint(10, 99), random.randint(10, 99)
    return f"{a} + {b}", str(a + b)

def gen_sub2():
    a = random.randint(10, 99)
    b = random.randint(1, min(9, a-1))
    return f"{a} - {b}", str(a - b)

def gen_mul1():
    a, b = random.randint(10, 99), random.randint(2, 9)
    return f"{a} * {b}", str(a * b)

def gen_add3():
    a, b, c = random.randint(10, 99), random.randint(10, 99), random.randint(10, 99)
    return f"{a} + {b} + {c}", str(a + b + c)

LEVEL_GENS = {
    1: [(gen_add2, 1)],
    2: [(gen_add2, 0.7), (gen_sub2, 0.3)],
    3: [(gen_add2, 0.5), (gen_sub2, 0.3), (gen_mul1, 0.2)],
    4: [(gen_add2, 0.4), (gen_sub2, 0.3), (gen_mul1, 0.2), (gen_add3, 0.1)]
}

def generate_dataset(level, n):
    examples = []
    level_mix = LEVEL_GENS.get(level, LEVEL_GENS[1])

    for _ in range(n):
        gen_fn = random.choices([fn for fn, _ in level_mix], weights=[prob for _, prob in level_mix], k=1)[0]
        expr, ans = gen_fn()
        examples.append(f"{expr} = {ans}")
    return examples

class MathDataset(torch.utils.data.Dataset):
    def __init__(self, examples, tokenizer, max_len=80):
        self.examples = [tokenizer.encode(ex, add_bos=True, add_eos=True) for ex in examples]
    def __len__(self): return len(self.examples)
    def __getitem__(self, i): return self.examples[i]

def collate(batch, pad_id):
    L = max(len(x) for x in batch)
    ids = torch.tensor([x + [pad_id] * (L - len(x)) for x in batch], dtype=torch.long)
    masks = torch.tensor([[1]*len(x) + [0]*(L-len(x)) for x in batch], dtype=torch.long)
    return ids, masks

# ============================================================
# DPPU CELL -- YOUR ORIGINAL CORE V13 (REPAIRED)
# ============================================================
class DPPUCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, d_cap=D_CAP):
        super().__init__()
        self.hidden_dim, self.d_cap, self.n_dims = hidden_dim, d_cap, d_cap + 1
        self.dim_size = hidden_dim // self.n_dims
        self.W_x = nn.Linear(input_dim, hidden_dim)
        self.W_h = nn.ModuleList([nn.Linear(hidden_dim, self.dim_size, bias=False) for _ in range(self.n_dims)])
        self.W_c = nn.Linear(hidden_dim, self.n_dims, bias=False)
        self.W_out = nn.Linear(hidden_dim, hidden_dim)

    def init_state(self, batch, device):
        return (torch.zeros(batch, self.hidden_dim, device=device),
                torch.zeros(batch, self.n_dims, device=device))

    def forward(self, x, h, C):
        x_proj = self.W_x(x)
        dim_outputs, phi_list, delta_list, pi_list = [], [], [], []

        for i in range(self.n_dims):
            s, e = i * self.dim_size, (i + 1) * self.dim_size
            h_i_prev = h[:, s:e]
            delta_i = compute_delta(h_i_prev)
            phi_i, pi_i, omg_i = phi_dyn(delta_i), pi_dyn(delta_i), omega(delta_i)

            # Dimensional Residual
            h_rec = (phi_i / PHI_CL) * self.W_h[i](h)
            x_rec = (pi_i / PI_CL) * x_proj[:, s:e]
            h_i_new = torch.tanh(h_rec + x_rec + C[:, i:i+1] + h_i_prev) * torch.sigmoid(omg_i)

            dim_outputs.append(torch.nan_to_num(h_i_new))
            phi_list.append(phi_i.mean().item())
            delta_list.append(delta_i.mean().item())
            pi_list.append(pi_i.mean().item())

        h_cat = torch.cat(dim_outputs, dim=-1)
        h_new = torch.nan_to_num(self.W_out(h_cat))

        # Consciousness field logic
        C_proj = self.W_c(h_new)
        delta_C = torch.stack([compute_delta(h_new[:, i*self.dim_size:(i+1)*self.dim_size]).mean(dim=-1) for i in range(self.n_dims)], dim=-1)
        C_new = torch.tanh(0.1 * C * pi_dyn(delta_C) * phi_dyn(delta_C) + 0.1 * torch.tanh(C_proj))

        metrics = {
            'phi_mean': sum(phi_list)/len(phi_list),
            'delta_mean': sum(delta_list)/len(delta_list),
            'pi_mean': sum(pi_list)/len(pi_list),
            'C_norm': C_new.norm().item()
        }
        return h_new, torch.nan_to_num(C_new), metrics

class VRUModel(nn.Module):
    def __init__(self, vocab_size, hidden_dim, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.cells = nn.ModuleList([DPPUCell(hidden_dim, hidden_dim) for _ in range(num_layers)])
        self.out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, token_ids, states=None):
        B, T = token_ids.shape
        x = self.embedding(token_ids)
        if states is None: states = [cell.init_state(B, token_ids.device) for cell in self.cells]
        new_states = []

        last_layer_metrics = None

        for i, cell in enumerate(self.cells):
            h, C = states[i]
            outputs = []

            for t in range(T):
                h, C, metrics = cell(x[:, t], h, C)
                outputs.append(h)
                if i == len(self.cells) - 1 and t == T - 1:
                    last_layer_metrics = metrics
            x = torch.stack(outputs, dim=1)
            new_states.append((h, C))
        return self.out(x), new_states, last_layer_metrics

# ============================================================
# REPAIRED TRAINING LOOP (MASTERY-BASED)
# ============================================================
def train():
    tokenizer = MathTokenizer()
    model = VRUModel(tokenizer.vocab_size, 260).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=200)

    print(f"Igniting Engine v13 Core on {device} (Mastery Mode)...")

    current_level = 1
    total_epochs = 300 # Give it plenty of time to master levels

    for epoch in range(1, total_epochs + 1):
        model.train()
        examples = generate_dataset(current_level, 100)
        ds = MathDataset(examples, tokenizer)
        loader = torch.utils.data.DataLoader(ds, batch_size=32, collate_fn=lambda b: collate(b, tokenizer.pad_id))

        epoch_loss, epoch_acc, total_tokens = 0.0, 0.0, 0
        epoch_phi, epoch_delta, epoch_pi, epoch_c = 0.0, 0.0, 0.0, 0.0

        for ids, mask in loader:
            ids, mask = ids.to(device), mask.to(device)
            inp, tgt = ids[:, :-1], ids[:, 1:]
            opt.zero_grad()

            logits, _, metrics = model(inp)
            loss = F.cross_entropy(logits.reshape(-1, tokenizer.vocab_size), tgt.reshape(-1), ignore_index=tokenizer.pad_id)
            loss.backward()
            opt.step()

            epoch_loss += loss.item()

            preds = logits.argmax(dim=-1)
            valid_mask = (tgt != tokenizer.pad_id)
            correct = (preds == tgt) & valid_mask
            epoch_acc += correct.sum().item()
            total_tokens += valid_mask.sum().item()

            if metrics:
                epoch_phi += metrics['phi_mean']
                epoch_delta += metrics['delta_mean']
                epoch_pi += metrics['pi_mean']
                epoch_c += metrics['C_norm']

        scheduler.step()

        # Reporting & Evaluation
        if epoch % 5 == 0 or epoch == 1:
            avg_loss = epoch_loss / len(loader)
            avg_acc = epoch_acc / total_tokens if total_tokens > 0 else 0.0
            avg_phi = epoch_phi / len(loader)
            avg_delta = epoch_delta / len(loader)
            avg_pi = epoch_pi / len(loader)
            avg_c = epoch_c / len(loader)

            model.eval()

            # MASTERY CHECK: Advance if Accuracy > 85%
            if avg_acc > 0.85 and current_level < len(LEVEL_GENS):
                print(f"\n*** MASTERY ACHIEVED ({avg_acc:.2%}). Advancing to Level {current_level + 1} ***\n")
                current_level += 1

            # Probe
            test_ex_input = "45 + 10 = "
            test_ex_target_ans = str(45 + 10)
            full_test_seq = test_ex_input + test_ex_target_ans
            test_ids = torch.tensor([tokenizer.encode(full_test_seq, add_bos=True, add_eos=True)], device=device)
            inp_probe, tgt_probe = test_ids[:, :-1], test_ids[:, 1:]

            with torch.no_grad():
                logits_probe, _, _ = model(inp_probe)
                test_loss = F.cross_entropy(logits_probe.reshape(-1, tokenizer.vocab_size), tgt_probe.reshape(-1), ignore_index=tokenizer.pad_id).item()

                predictions_probe = logits_probe.argmax(dim=-1)
                non_pad_mask_probe = (tgt_probe != tokenizer.pad_id)
                correct_predictions_probe = (predictions_probe == tgt_probe) & non_pad_mask_probe
                probe_accuracy = correct_predictions_probe.sum().item() / non_pad_mask_probe.sum().item() if non_pad_mask_probe.sum().item() > 0 else 0.0

                # Generation
                generated_answer_ids = []
                current_input_ids_for_gen = tokenizer.encode(test_ex_input, add_bos=True, add_eos=False)
                current_h_states_for_gen = [cell.init_state(1, device) for cell in model.cells]

                for t_gen in range(20):
                    input_for_next_cell = model.embedding(torch.tensor([[current_input_ids_for_gen[-1]]], device=device)).squeeze(0)
                    new_h_states_gen = []
                    for l_idx, cell in enumerate(model.cells):
                        h_gen, C_gen = current_h_states_for_gen[l_idx]
                        h_new_gen, C_new_gen, _ = cell(input_for_next_cell, h_gen, C_gen)
                        input_for_next_cell = h_new_gen
                        new_h_states_gen.append((h_new_gen, C_new_gen))
                    current_h_states_for_gen = new_h_states_gen
                    next_token_logits = model.out(input_for_next_cell)
                    next_token_id = next_token_logits.argmax(dim=-1).item()
                    if next_token_id == tokenizer.eos_id: break
                    generated_answer_ids.append(next_token_id)
                    current_input_ids_for_gen.append(next_token_id)

                predicted_answer_str = tokenizer.decode(generated_answer_ids, skip_special=True)

            print(f"Epoch {epoch} (Level {current_level}) | Train Loss: {avg_loss:.4f} | Train Acc: {avg_acc:.4f}")
            print(f"    Field Sustainability: Phi={avg_phi:.4f}, Delta={avg_delta:.4f}, Pi={avg_pi:.4f}, C={avg_c:.4f}")
            print(f"    Probe '{test_ex_input}': Predicted='{predicted_answer_str}', Expected='{test_ex_target_ans}'")
            print(f"    Probe Test Loss: {test_loss:.4f} | Probe Acc: {probe_accuracy:.4f}")

    print("Training Complete.")

if __name__ == "__main__":
    train()


Igniting Engine v13 Core on cuda...
Epoch 1 (Level 1) | Train Loss: 2.7489 | Train Acc: 0.2599
    Field Sustainability (Train): Phi=1.1530, Delta=0.6263, Pi=3.5195, C=0.2259
    Probe '45 + 10 = ': Predicted='= 1                 ', Expected='55'
    Probe Test Loss: 1.9342 | Probe Acc: 0.3846
Epoch 5 (Level 1) | Train Loss: 1.1405 | Train Acc: 0.6103
    Field Sustainability (Train): Phi=1.1528, Delta=0.6281, Pi=3.5201, C=0.5879
    Probe '45 + 10 = ': Predicted='= 111', Expected='55'
    Probe Test Loss: 1.0477 | Probe Acc: 0.6154
Epoch 10 (Level 1) | Train Loss: 1.0850 | Train Acc: 0.6049
    Field Sustainability (Train): Phi=1.1527, Delta=0.6289, Pi=3.5204, C=0.7381
    Probe '45 + 10 = ': Predicted='= 122', Expected='55'
    Probe Test Loss: 1.0736 | Probe Acc: 0.6154
Epoch 15 (Level 1) | Train Loss: 1.0621 | Train Acc: 0.6122
    Field Sustainability (Train): Phi=1.1527, Delta=0.6282, Pi=3.5203, C=0.8347
    Probe '45 + 10 = ': Predicted='= 117', Expected='55'
    Probe Test Loss

KeyboardInterrupt: 